# 🎬 07 — Recommendations Analysis
Visualizes and examines recommendation recommendations for users. Identifies 5 success cases (with taste analyses), 3 failure cases (with diagnostic notes), and computes popularity bias on the top recommendations across 100 power users.

In [1]:
import sys
import os
from pathlib import Path
sys.path.append(str(Path(os.getcwd()).parent))
import pandas as pd
import numpy as np
from src.models.svd_model import load_model
from src.data_pipeline import load_movie_titles
from src.recommender import get_top_k_recommendations, get_user_history, explain_recommendation
from collections import Counter

# Load data
train_df = pd.read_parquet('../data/processed/train.parquet')
movies = load_movie_titles()
svd_model = load_model('../models/svd_model.pkl')
all_movie_ids = train_df['movie_id'].unique().tolist()

# Find power users
power_users = train_df['user_id'].value_counts().head(100).index.tolist()

# SUCCESS CASES (5 Users)
print("SUCCESS CASES (5 Users):")
for idx, user_id in enumerate(power_users[:5]):
    print(f"\n{'='*60}")
    print(f"USER {user_id} - Top Rated Movies in History:")
    history = get_user_history(user_id, train_df, movies, n=10)
    print(history.to_string(index=False))
    
    print(f"\nTop-10 Recommendations for User {user_id}:")
    seen = set(train_df[train_df['user_id'] == user_id]['movie_id'])
    recs = get_top_k_recommendations(svd_model, user_id, all_movie_ids, seen, movies, k=10)
    print(recs.to_string(index=False))
    
    print(f"\nTaste Analysis & Explainability:")
    print(explain_recommendation(user_id, recs['movie_id'].iloc[0], svd_model, train_df, movies))

# FAILURE CASES (3 Users)
print(f"\n{'='*60}")
print("DIAGNOSING FAILURE CASES:")
few_ratings_users = train_df['user_id'].value_counts().tail(5).index.tolist()
for user_id in few_ratings_users[:3]:
    print(f"\nUSER {user_id} (Low Activity - Ratings Count = {len(train_df[train_df['user_id'] == user_id])}):")
    history = get_user_history(user_id, train_df, movies, n=5)
    print(history.to_string(index=False))
    
    seen = set(train_df[train_df['user_id'] == user_id]['movie_id'])
    recs = get_top_k_recommendations(svd_model, user_id, all_movie_ids, seen, movies, k=5)
    print("Recommendations:")
    print(recs.to_string(index=False))
    print("Diagnosis: With very few ratings, the model relies heavily on global biases rather than personalized tastes. It recommends highly popular films with generic positive ratings. A content-based approach or active user preference elicitation is required to solve this.")

# POPULARITY BIAS
print(f"\n{'='*60}")
print("POPULARITY BIAS ANALYSIS:")
rec_movie_ids = []
for user_id in power_users[:100]:
    seen = set(train_df[train_df['user_id'] == user_id]['movie_id'])
    recs = get_top_k_recommendations(svd_model, user_id, all_movie_ids, seen, movies, k=10)
    rec_movie_ids.extend(recs['movie_id'].tolist() if 'movie_id' in recs.columns else [])

rec_freq = Counter(rec_movie_ids)
print("\nMost frequently recommended movies across 100 power users:")
top_rec_movies = pd.DataFrame(rec_freq.most_common(20), columns=['movie_id', 'times_recommended'])
top_rec_movies = top_rec_movies.merge(movies, on='movie_id')
print(top_rec_movies[['title', 'times_recommended']].to_string(index=False))

Loading movie titles from: D:\project_4th_year\cult_summer_project\netflix-recommendation-system\data\raw\movie_titles.csv


SUCCESS CASES (5 Users):

USER 305344 - Top Rated Movies in History:
 movie_id                                             title   year  rating
     9572      Alanis Morissette: Live in the Navajo Nation 2002.0       5
    13705                                    Basic Instinct 1992.0       5
     4736                                  Chariots of Fire 1981.0       5
    16325                                Planet of the Apes 1968.0       5
     5355                                    A Star Is Born 1937.0       5
     6142 Law & Order: Special Victims Unit: The Fifth Year 2003.0       5
     2429          Close to You: Remembering the Carpenters 1997.0       5
     4547                  The Three Stooges Double Feature 1947.0       5
    16377                                    The Green Mile 1999.0       5
    15671                                   The Running Man 1987.0       5

Top-10 Recommendations for User 305344:
 rank  movie_id                          title   year  predicted_

 rank  movie_id                                   title   year  predicted_rating
    1      2203 All Creatures Great and Small: Series 3 1980.0          3.425863
    2     16006                 Seinfeld: Seasons 1 & 2 1989.0          3.363131
    3      7833          Arrested Development: Season 2 2004.0          3.350609
    4     10778                        Scrubs: Season 1 2001.0          3.249062
    5      5738                         Alias: Season 3 2003.0          3.244456
    6     17085                            24: Season 2 2002.0          3.222516
    7      7569                  Dead Like Me: Season 2 2004.0          3.213100
    8      7683         Cinema Paradiso: Director's Cut 1988.0          3.212143
    9      9326                 Queer as Folk: Season 3 2003.0          3.208059
   10      7742                Six Feet Under: Season 3 2003.0          3.202625

Taste Analysis & Explainability:
We recommend 'All Creatures Great and Small: Series 3' because users who hi


USER 2373901 (Low Activity - Ratings Count = 1):
 movie_id title   year  rating
      273  Taxi 2004.0       3
Recommendations:
 rank  movie_id                                                               title   year  predicted_rating
    1      7230 The Lord of the Rings: The Fellowship of the Ring: Extended Edition 2001.0          4.668236
    2      7833                                      Arrested Development: Season 2 2004.0          4.653800
    3      7569                                              Dead Like Me: Season 2 2004.0          4.619327
    4     14961         Lord of the Rings: The Return of the King: Extended Edition 2003.0          4.614401
    5      5103                                              The Simpsons: Season 5 1993.0          4.587387
Diagnosis: With very few ratings, the model relies heavily on global biases rather than personalized tastes. It recommends highly popular films with generic positive ratings. A content-based approach or active user pr


Most frequently recommended movies across 100 power users:
                                                              title  times_recommended
                                     Arrested Development: Season 2                 80
The Lord of the Rings: The Fellowship of the Ring: Extended Edition                 65
        Lord of the Rings: The Return of the King: Extended Edition                 63
                                             Dead Like Me: Season 2                 62
                            All Creatures Great and Small: Series 3                 47
                                             The Sopranos: Season 5                 44
                                             The Simpsons: Season 5                 34
                Lord of the Rings: The Two Towers: Extended Edition                 32
                                                 The Wire: Season 2                 21
                                            The West Wing: Season 3   